# Comparativa Featured-HMM vs TDE-HMM

In [ ]:
import sys, json, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, yaml
from pathlib import Path
from scipy import stats
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120})

PROJECT_NAME = "eeg_hmm_plattform"
current = Path.cwd()
while current.name != PROJECT_NAME:
    if current.parent == current:
        raise RuntimeError(f"No se encontro {PROJECT_NAME}")
    current = current.parent
PROJECT_ROOT = current

# Experimento Featured a comparar (principal)
FEATURED_YAML = "canonical/canonical_k4_full_95d_v2.yaml"

# Experimento TDE a comparar (principal — cambiar cuando esten los resultados HPC)
TDE_YAML = "canonical/tde_k4_t7.yaml"

STATE_COLORS = ["#E63946","#2196F3","#4CAF50","#FF9800","#9C27B0","#00BCD4"]
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "comparative"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print(f"Featured YAML : {FEATURED_YAML}")
print(f"TDE YAML      : {TDE_YAML}")


---
## Carga de ambos experimentos

In [ ]:
def load_exp(yaml_rel):
    yaml_path = PROJECT_ROOT / "configs" / "experiments" / yaml_rel
    if not yaml_path.exists():
        print(f"  no encontrado: {yaml_path}")
        return None
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    exp_name      = cfg["experiment"]["name"]
    K             = cfg["pipeline"]["hmm"]["k_states"]
    pipeline_type = cfg["experiment"].get("pipeline_type", "feature")

    output_dir = PROJECT_ROOT / cfg["paths"]["output_dir"].replace("../../","").replace("../","")
    exp_dir    = output_dir / exp_name

    required = [exp_dir / f"hmm_model_k{K}.pkl",
                exp_dir / f"viterbi_paths_k{K}.npy",
                exp_dir / "decoding_summary.json",
                exp_dir / "state_stats.csv",
                exp_dir / "fo_by_subject.csv",
                exp_dir / "lengths.npy"]
    missing = [r.name for r in required if not r.exists()]
    if missing:
        print(f"  SKIP {exp_name}: faltan {missing}")
        return None

    model     = joblib.load(exp_dir / f"hmm_model_k{K}.pkl")
    viterbi   = np.load(exp_dir / f"viterbi_paths_k{K}.npy")
    X_pca     = np.load(exp_dir / "X_pca.npy")
    lengths   = np.load(exp_dir / "lengths.npy")
    posterior = model.predict_proba(X_pca)
    summary   = json.loads((exp_dir / "decoding_summary.json").read_text())
    df_stats  = pd.read_csv(exp_dir / "state_stats.csv")
    df_fo     = pd.read_csv(exp_dir / "fo_by_subject.csv")
    manifest  = json.loads((exp_dir / "feature_manifest.json").read_text())

    if pipeline_type == "tde":
        sfreq   = cfg["tde"].get("sfreq", 250.0)
        step_ms = 1000.0 / sfreq
        n_lags  = cfg["tde"]["n_lags"]
    else:
        step_ms = float(cfg.get("windowing",{}).get("step_ms", 100))
        n_lags  = None

    print(f"  OK {exp_name} | pipeline={pipeline_type} | K={K} | score={summary[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)]}/4")
    return {
        "name":      exp_name,
        "pipeline":  pipeline_type,
        "K":         K,
        "n_lags":    n_lags,
        "step_ms":   step_ms,
        "model":     model,
        "viterbi":   viterbi,
        "X_pca":     X_pca,
        "lengths":   lengths,
        "posterior": posterior,
        "summary":   summary,
        "df_stats":  df_stats,
        "df_fo":     df_fo,
        "manifest":  manifest,
        "exp_dir":   exp_dir,
        "cfg":       cfg,
    }

print("Cargando...")
feat = load_exp(FEATURED_YAML)
tde  = load_exp(TDE_YAML)

if feat is None or tde is None:
    print()
    print("Uno o ambos experimentos no estan disponibles.")
    print("Ejecuta los pipelines primero y vuelve a correr este notebook.")


---
## Tabla comparativa de metricas globales

In [ ]:
if feat and tde:
    rows = []
    for e in [feat, tde]:
        s = e["summary"]
        rows.append({
            "Metodo":       e["pipeline"].upper(),
            "Experimento":  e["name"],
            "K":            e["K"],
            "Score":        s[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)],
            "FO_min":       s["FO_min"],
            "FO_max":       s["FO_max"],
            "dwell_max_ms": s["dwell_max_ms"],
            "confidence":   s["confidence"],
            "n_absorbing":  s["n_absorbing"],
            "Veredicto":    s[chr(118)+chr(101)+chr(114)+chr(100)+chr(105)+chr(99)+chr(116)],
        })
    df_cmp = pd.DataFrame(rows)
    display(Markdown("### Comparativa global"))
    display(df_cmp.style
        .background_gradient(subset=["FO_min"], cmap="RdYlGn")
        .background_gradient(subset=["dwell_max_ms"], cmap="RdYlGn_r")
        .background_gradient(subset=["confidence"], cmap="Greens")
        .format({"FO_min":"{:.4f}","FO_max":"{:.4f}",
                 "dwell_max_ms":"{:.1f}","confidence":"{:.4f}"})
    )


---
## Comparativa FO por condicion y grupo

In [ ]:
if feat and tde:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
    methods = [(feat, 0), (tde, 1)]

    for e, row_i in methods:
        for col_i, (cond, group) in enumerate([
            ("GO","ADULTO"),("NOGO","ADULTO"),
            ("GO","ADOLESCENTE"),("NOGO","ADOLESCENTE")
        ]):
            ax  = axes[row_i, col_i]
            sub = e["df_fo"][(e["df_fo"]["condition"]==cond) &
                              (e["df_fo"]["group"]==group)]
            if sub.empty:
                ax.text(0.5,0.5,"sin datos",ha="center",va="center",transform=ax.transAxes)
                continue
            means = sub.groupby("state")["FO"].mean()
            sems  = sub.groupby("state")["FO"].sem()
            x     = np.arange(e["K"])
            ax.bar(x, [means.get(f"S{s}",0) for s in range(e["K"])],
                   yerr=[sems.get(f"S{s}",0) for s in range(e["K"])],
                   color=STATE_COLORS[:e["K"]], alpha=0.85, capsize=4)
            ax.axhline(1/e["K"], color="gray", linestyle="--", alpha=0.4)
            ax.set_xticks(x); ax.set_xticklabels([f"S{s}" for s in range(e["K"])])
            if col_i == 0:
                ax.set_ylabel(f"{e[chr(112)+chr(105)+chr(112)+chr(101)+chr(108)+chr(105)+chr(110)+chr(101)].upper()}
FO media")
            if row_i == 0:
                ax.set_title(f"{group}
{cond}", fontweight="bold")

    plt.suptitle("FO por condicion y grupo — Featured vs TDE",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); save_fig("fo_comparison"); plt.show()


---
## Comparativa de dwell times

In [ ]:
if feat and tde:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, col, title in zip(axes,
        ["dwell_analytic_ms","dwell_median_ms"],
        ["Dwell analitico (ms)","Dwell mediana (ms)"]):
        for e, ls in [(feat,"-"),(tde,"--")]:
            states = [f"S{s}" for s in range(e["K"])]
            vals   = [e["df_stats"][e["df_stats"]["state"]==s][col].values[0]
                      if s in e["df_stats"]["state"].values else 0
                      for s in states]
            ax.plot(states, vals, marker="o", lw=2.5, ls=ls,
                    label=e["pipeline"].upper(), alpha=0.85)
        ax.axhline(2000, color="red", linestyle=":", alpha=0.5, label="limite 2s")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Estado"); ax.set_ylabel("ms"); ax.legend()
    plt.suptitle("Dwell times — Featured vs TDE", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_fig("dwell_comparison"); plt.show()


---
## Comparativa de absorbentes

In [ ]:
if feat and tde:
    abs_feat = {a["subject_id"] for a in feat["summary"].get("absorbing",[])}
    abs_tde  = {a["subject_id"] for a in tde["summary"].get("absorbing",[])}

    solo_feat  = abs_feat - abs_tde
    solo_tde   = abs_tde  - abs_feat
    en_ambos   = abs_feat & abs_tde

    print("Absorbentes:")
    print(f"  Featured solo : {len(solo_feat)}")
    print(f"  TDE solo      : {len(solo_tde)}")
    print(f"  En ambos      : {len(en_ambos)}")
    if en_ambos:
        print(f"  Sujetos en ambos: {sorted(en_ambos)}")

    sizes = [len(solo_feat), len(en_ambos), len(solo_tde)]
    labels = [f"Solo Featured
(n={len(solo_feat)})",
              f"Ambos
(n={len(en_ambos)})",
              f"Solo TDE
(n={len(solo_tde)})"]
    if any(s > 0 for s in sizes):
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(labels, sizes, color=["#E63946","#9C27B0","#2196F3"], alpha=0.85)
        for bar, val in zip(bars, sizes):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                    str(val), ha="center", fontweight="bold")
        ax.set_ylabel("N sesiones absorbentes")
        ax.set_title("Absorbentes: consistencia entre metodos", fontweight="bold")
        plt.tight_layout(); save_fig("absorbing_comparison"); plt.show()


---
## Interpretacion comparativa

In [ ]:
if feat and tde:
    print("="*60)
    print("COMPARATIVA Featured-HMM vs TDE-HMM")
    print("="*60)

    for e in [feat, tde]:
        s = e["summary"]
        print(f"
{e[chr(112)+chr(105)+chr(112)+chr(101)+chr(108)+chr(105)+chr(110)+chr(101)].upper()} — {e[chr(110)+chr(97)+chr(109)+chr(101)]}")
        print(f"  Score     : {s[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)]}/4 — {s[chr(118)+chr(101)+chr(114)+chr(100)+chr(105)+chr(99)+chr(116)]}")
        print(f"  FO range  : {s[chr(70)+chr(79)+chr(95)+chr(109)+chr(105)+chr(110)]:.3f} - {s[chr(70)+chr(79)+chr(95)+chr(109)+chr(97)+chr(120)]:.3f}")
        print(f"  Dwell max : {s[chr(100)+chr(119)+chr(101)+chr(108)+chr(108)+chr(95)+chr(109)+chr(97)+chr(120)+chr(95)+chr(109)+chr(115)]:.1f}ms")
        print(f"  Confianza : {s[chr(99)+chr(111)+chr(110)+chr(102)+chr(105)+chr(100)+chr(101)+chr(110)+chr(99)+chr(101)]:.4f}")
        print(f"  Absorbentes: {s[chr(110)+chr(95)+chr(97)+chr(98)+chr(115)+chr(111)+chr(114)+chr(98)+chr(105)+chr(110)+chr(103)]}/{s[chr(110)+chr(95)+chr(115)+chr(101)+chr(115)+chr(115)+chr(105)+chr(111)+chr(110)+chr(115)]}")

    print()
    print("Preguntas clave para la tesis:")
    print("  1. Los estados TDE tienen mejor resolucion topografica?")
    print("  2. La dinamica evocada Go/NoGo se reproduce en TDE?")
    print("  3. Los mismos sujetos son absorbentes en ambos metodos?")
    print("  4. El K optimo es el mismo en ambos metodos?")
else:
    print("Ejecuta primero los experimentos en la HPC y vuelve a correr este notebook.")
    print(f"Pendiente: {TDE_YAML}")
